# 11-3. Poisson QMLE 종일 최종모형 튜닝

이 노트북은 `11-2` 음이항 모형과 같은 종일 셀×노선 표본, 같은 offset, 같은 노선 고정효과를 사용한다.

- 종속변수: 2025년 종일 승차인원 `Y2025`
- offset: `log(2024년 종일 승차인원)`
- 분석 단위: 셀 × 노선
- 표준오차: 셀 단위 군집 강건 표준오차
- 목적지 비율: 오전 파일이 아니라 **2024년 종일 서울 하차 기준 파일** 사용

Poisson QMLE는 분산이 평균보다 큰 상황에서도 평균 구조를 추정하고, 군집 강건 표준오차로 추론하는 본모형 후보이다. 아래에서 기본모형과 변수 조합을 비교한다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from config import *
setup_notebook()

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

data = load_model_data()
f = load_cell_route_all_day()

print('목적지 비율 파일:', COMPETING_PATH)
print('종일 목적지 비율 평균:', round(data['competing_share'].mean(), 4))
print('종일 목적지 비율 결측:', int(data['competing_share'].isna().sum()))

f['셀 ID'] = f['셀 ID'].astype(int)
f = f.merge(data[['셀 ID'] + X_MAIN + ['walk_20', 'walk_20_60']], on='셀 ID', how='inner')
f = f[f['y2024'] > 0].reset_index(drop=True)

print(f'분석 표본: {len(f)}개 셀×노선')
print(f'분석 셀 수: {f["셀 ID"].nunique()}개 | 노선 수: {f["노선"].nunique()}개')
print(f'2024 평균·분산: {f["y2024"].mean():.2f}, {f["y2024"].var(ddof=1):.2f}')
print(f'2025 평균·분산: {f["y2025"].mean():.2f}, {f["y2025"].var(ddof=1):.2f}')

목적지 비율 파일: C:\Users\금경훈\Desktop\Gachon\3-2\UROP\github_2026-UROP\gtx_a_bus_analysis\data\processed\06_GTX_경쟁목적지비율_종일.csv
종일 목적지 비율 평균: 12.6176
종일 목적지 비율 결측: 0
분석 표본: 220개 셀×노선
분석 셀 수: 132개 | 노선 수: 24개
2024 평균·분산: 90.38, 23810.84
2025 평균·분산: 74.09, 21581.11


In [2]:
# Poisson QMLE 적합 함수
BASE_X = ['walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
CONTINUOUS = ['ic_access', 'competing_share', 'pop_20_50', 'buildings']

def make_design(frame, x_cols, standardize=True, route_fe=True):
    x = frame[x_cols].copy().astype(float)
    if standardize:
        for col in CONTINUOUS:
            if col in x.columns:
                sd = x[col].std(ddof=0)
                x[col] = (x[col] - x[col].mean()) / (sd if sd else 1)
    if route_fe:
        route = pd.get_dummies(frame['노선'], prefix='노선', drop_first=True, dtype=float)
        x = pd.concat([x, route], axis=1)
    return sm.add_constant(x, has_constant='add')

def fit_poisson_qmle(x_cols, standardize=True, route_fe=True):
    X = make_design(f, x_cols, standardize=standardize, route_fe=route_fe)
    model = sm.GLM(
        f['y2025'], X,
        family=sm.families.Poisson(),
        offset=np.log(f['y2024'])
    )
    result = model.fit(
        cov_type='cluster',
        cov_kwds={'groups': pd.factorize(f['셀 ID'])[0], 'use_correction': True}
    )
    dispersion = result.pearson_chi2 / result.df_resid
    bic_llf = -2 * result.llf + len(result.params) * np.log(result.nobs)
    pseudo_r2 = 1 - result.deviance / result.null_deviance if result.null_deviance else np.nan
    return result, {
        '표본 수': int(result.nobs),
        '로그우도': result.llf,
        'AIC': result.aic,
        'BIC(로그우도)': bic_llf,
        'deviance pseudo R²': pseudo_r2,
        'Pearson dispersion': dispersion,
        '노선 고정효과': route_fe,
        '표준화': standardize,
    }

In [3]:
# 1. 기본 Poisson QMLE 모형
pois_fit, pois_info = fit_poisson_qmle(BASE_X)
print(pois_fit.summary())
print('\n===== 기본모형 적합도 =====')
display(pd.DataFrame([pois_info]).round(4))

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  y2025   No. Observations:                  220
Model:                            GLM   Df Residuals:                      190
Model Family:                 Poisson   Df Model:                           29
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1733.1
Date:                Mon, 05 Oct 2026   Deviance:                       2321.0
Time:                        13:45:57   Pearson chi2:                 2.70e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.9999
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.0375      0.171     

,표본 수,로그우도,AIC,BIC(로그우도),deviance pseudo R²,Pearson dispersion,노선 고정효과,표준화
0,220,-1733.1043,3526.2085,3628.0173,0.4778,14.2229,True,True


In [4]:
# 2. 변수 구성별 튜닝 비교
specs = {
    '기본모형_전체변수': {'x_cols': BASE_X, 'standardize': True, 'route_fe': True},
    '핵심변수_보행_IC': {'x_cols': ['walk_20', 'walk_20_60', 'ic_access'], 'standardize': True, 'route_fe': True},
    '건축물_제외': {'x_cols': ['walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'pop_20_50'], 'standardize': True, 'route_fe': True},
    '경쟁목적지_제외': {'x_cols': ['walk_20', 'walk_20_60', 'ic_access', 'pop_20_50', 'buildings'], 'standardize': True, 'route_fe': True},
    '인구·건축물_제외': {'x_cols': ['walk_20', 'walk_20_60', 'ic_access', 'competing_share'], 'standardize': True, 'route_fe': True},
    '노선고정효과_제외': {'x_cols': BASE_X, 'standardize': True, 'route_fe': False},
    '표준화_제외': {'x_cols': BASE_X, 'standardize': False, 'route_fe': True},
}

fits = {}
rows = []
for name, spec in specs.items():
    try:
        fit, info = fit_poisson_qmle(**spec)
        fits[name] = fit
        info = {'모형': name, **info}
        rows.append(info)
    except Exception as exc:
        rows.append({'모형': name, '오류': repr(exc)})

tuning_table = pd.DataFrame(rows)
display(tuning_table.round(4))

,모형,표본 수,로그우도,AIC,BIC(로그우도),deviance pseudo R²,Pearson dispersion,노선 고정효과,표준화
0,기본모형_전체변수,220,-1733.1043,3526.2085,3628.0173,0.4778,14.2229,True,True
1,핵심변수_보행_IC,220,-1791.7595,3637.5191,3729.1470,0.4514,14.3934,True,True
2,건축물_제외,220,-1734.5752,3527.1503,3625.5655,0.4771,14.1900,True,True
3,경쟁목적지_제외,220,-1737.7921,3533.5842,3631.9994,0.4757,14.2489,True,True
4,인구·건축물_제외,220,-1787.8956,3631.7912,3726.8128,0.4531,14.3844,True,True
5,노선고정효과_제외,220,-2499.2232,5012.4464,5036.2018,0.1331,21.9656,False,True
6,표준화_제외,220,-1733.1043,3526.2085,3628.0173,0.4778,14.2229,True,False


In [5]:
# 3. 기본모형 변수별 효과·p값·95% 신뢰구간
continuous_labels = {
    'ic_access': 'IC까지 도로거리',
    'competing_share': 'GTX 경쟁 목적지 비율',
    'pop_20_50': '20~50대 인구',
    'buildings': '건축물 수',
}
label = {**VARIABLE_LABELS, **continuous_labels}
ci = pois_fit.conf_int()
target = ['walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'pop_20_50', 'buildings']
effect = pd.DataFrame({
    '변수': [label.get(c, c) for c in target],
    '계수': [pois_fit.params.get(c, np.nan) for c in target],
    '상대적 변화효과(%)': [(np.exp(pois_fit.params.get(c, np.nan)) - 1) * 100 for c in target],
    '95% 하한(%)': [(np.exp(ci.loc[c, 0]) - 1) * 100 if c in ci.index else np.nan for c in target],
    '95% 상한(%)': [(np.exp(ci.loc[c, 1]) - 1) * 100 if c in ci.index else np.nan for c in target],
    'p-value': [pois_fit.pvalues.get(c, np.nan) for c in target],
})
display(effect.round({'계수': 4, '상대적 변화효과(%)': 2, '95% 하한(%)': 2, '95% 상한(%)': 2, 'p-value': 4}))

,변수,계수,상대적 변화효과(%),95% 하한(%),95% 상한(%),p-value
0,보행 20분 이내,-0.9203,-60.16,-72.38,-42.54,0.0000
1,보행 20~60분,-0.2850,-24.80,-40.17,-5.47,0.0146
2,IC까지 도로거리,-0.1487,-13.82,-23.72,-2.63,0.0170
3,GTX 경쟁 목적지 비율,-0.0674,-6.51,-26.22,18.45,0.5769
4,20~50대 인구,0.0978,10.27,-3.18,25.58,0.1406
5,건축물 수,-0.0205,-2.03,-13.02,10.35,0.7351


## 해석 메모

- `walk_20`과 `walk_20_60`의 기준집단은 보행시간 60분 초과 셀이다.
- 계수는 offset인 2024년 승차인원을 기준으로 한 2025년 승차 변화율의 로그 차이이다.
- `exp(계수)-1`을 백분율로 바꾸면 상대적 승차 변화효과가 된다.
- Poisson QMLE의 핵심 비교 기준은 계수 방향, 군집 강건 p-value, 효과크기, 그리고 모형 적합도이다.
- NB2 결과는 `11-2`에서 같은 종일 자료와 같은 목적지 비율을 사용해 강건성 분석으로 비교한다.

## 추가모형: 2024~2025년 20~50대 인구 증감 변수 적용

In [1]:
# Population-change replacement Poisson QMLE model (all-day)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import *
setup_notebook()
import statsmodels.api as sm
from IPython.display import display
ID = '\uc140 ID'; ROUTE = '\ub178\uc120'
POP_FEATURE = 'pop_change'
FEATURES = ['walk_20', 'walk_20_60', 'ic_access', 'competing_share', POP_FEATURE, 'buildings']
base = load_model_data(); routes = load_cell_route_all_day()
base_cols = [ID, 'walk_20', 'walk_20_60', 'ic_access', 'competing_share', 'buildings']
f_pop = routes.merge(base[base_cols], on=ID, how='inner', validate='many_to_one')
pop = read_csv(POPULATION_CHANGE_PATH); pop[ID] = pd.to_numeric(pop[ID], errors='coerce').astype('Int64')
pop_col = next(c for c in pop.columns if '증감(100명)' in str(c))

pop = pop[[ID, pop_col]].copy()
pop[pop_col] = pd.to_numeric(pop[pop_col], errors='coerce')

# 기존 자료가 '100명 단위'이므로 실제 인구수(명)로 변환
pop['pop_change'] = pop[pop_col] * 100

pop = pop[[ID, 'pop_change']]
f_pop = f_pop.merge(pop, on=ID, how='inner', validate='many_to_one')
f_pop = f_pop[f_pop['y2024'] > 0].dropna(subset=['y2024', 'y2025'] + FEATURES).reset_index(drop=True)
x_pop = f_pop[FEATURES].astype(float).copy()
route_dum = pd.get_dummies(f_pop[ROUTE], prefix='route', drop_first=True, dtype=float)
X_pop = sm.add_constant(pd.concat([x_pop, route_dum], axis=1), has_constant='add'); off_pop = np.log(f_pop['y2024'])
pois_pop_fit = sm.GLM(f_pop['y2025'], X_pop, family=sm.families.Poisson(), offset=off_pop).fit(cov_type='cluster', cov_kwds={'groups': pd.factorize(f_pop[ID])[0], 'use_correction': True})
print('population-change Poisson QMLE:', len(f_pop), 'cell-route observations')
print(pois_pop_fit.summary())
bic_llf_pop = -2 * pois_pop_fit.llf + len(pois_pop_fit.params) * np.log(pois_pop_fit.nobs)
display(pd.DataFrame([{'model': 'population_change_Poisson_QMLE', 'n': len(f_pop), 'logLik': pois_pop_fit.llf, 'deviance_R2': 1 - pois_pop_fit.deviance / pois_pop_fit.null_deviance, 'AIC': pois_pop_fit.aic, 'BIC_llf': bic_llf_pop, 'Pearson_dispersion': pois_pop_fit.pearson_chi2 / pois_pop_fit.df_resid}]).round(4))
ci_pop = pois_pop_fit.conf_int().loc[FEATURES]
result_pop = pd.DataFrame({'variable': FEATURES, 'coef': [pois_pop_fit.params[v] for v in FEATURES], 'IRR': [np.exp(pois_pop_fit.params[v]) for v in FEATURES], 'effect_pct': [(np.exp(pois_pop_fit.params[v]) - 1) * 100 for v in FEATURES], 'z': [pois_pop_fit.params[v] / pois_pop_fit.bse[v] for v in FEATURES], 'p': [pois_pop_fit.pvalues[v] for v in FEATURES], 'CI_low_pct': [(np.exp(ci_pop.loc[v, 0]) - 1) * 100 for v in FEATURES], 'CI_high_pct': [(np.exp(ci_pop.loc[v, 1]) - 1) * 100 for v in FEATURES]})
display(result_pop.round(4)); print(
    'Population variable: 2024 to 2025 working-age population change, '
    'per person; no standardization; old population-level variable excluded.'
)

population-change Poisson QMLE: 220 cell-route observations
                 Generalized Linear Model Regression Results                  
Dep. Variable:                  y2025   No. Observations:                  220
Model:                            GLM   Df Residuals:                      190
Model Family:                 Poisson   Df Model:                           29
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1787.3
Date:                Mon, 05 Oct 2026   Deviance:                       2429.3
Time:                        17:36:19   Pearson chi2:                 2.76e+03
No. Iterations:                     7   Pseudo R-squ. (CS):             0.9999
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------

,model,n,logLik,deviance_R2,AIC,BIC_llf,Pearson_dispersion
0,population_change_Poisson_QMLE,220,-1787.2531,0.4534,3634.5063,3736.3151,14.5143


,variable,coef,IRR,effect_pct,z,p,CI_low_pct,CI_high_pct
0,walk_20,-0.9019,0.4058,-59.4197,-4.1817,0.0000,-73.4091,-38.0705
1,walk_20_60,-0.2954,0.7443,-25.5735,-2.5947,0.0095,-40.4565,-6.9705
2,ic_access,-0.0480,0.9532,-4.6835,-2.0249,0.0429,-9.0078,-0.1536
3,competing_share,-0.0031,0.9969,-0.3090,-0.5054,0.6133,-1.4985,0.8948
4,pop_change,-0.0001,0.9999,-0.0111,-0.2494,0.8030,-0.0986,0.0764
5,buildings,-0.0002,0.9998,-0.0159,-0.0609,0.9514,-0.5258,0.4967


Population variable: 2024 to 2025 working-age population change, per person; no standardization; old population-level variable excluded.
